# Stage A2 — CKA analysis

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
Computes **Linear CKA** between every layer pair of the two models. CKA
compares the *pairwise-similarity structure* of the representations (the
Gram matrix "fingerprint"), and is invariant to rotations, permutations and
isotropic scaling — exactly the symmetries under which a network's function
is preserved. So it is blind to meaningless coordinate differences and
sensitive to what is actually near what.

## How to read the result
- **Hypothesis supported:** the trained-vs-trained matrix shows a *hot
  diagonal* (early layers match early, middle match middle), while the
  trained-vs-random matrix is uniformly cold.
- **Hypothesis rejected:** both matrices look like structureless noise.

## Success criterion
Diagonal CKA mean > 0.5 with the random baseline < ~0.15 (a 3-4x gap).


In [ ]:
# Storage setup — where stage artifacts (.npz, .png) are read/written.
# Each stage reads the previous stage's output from DATA_DIR.
#
# Option 1 (default): current directory. Works if you run ALL stages in
# the SAME runtime/session. In Colab, a new notebook = a new VM, so files
# from a previous notebook are gone.
#
# Option 2 (Colab, persistent): mount Google Drive and point DATA_DIR
# there — artifacts survive across notebooks and sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

import os
os.environ.setdefault("DATA_DIR", ".")
print("DATA_DIR =", os.path.abspath(os.environ["DATA_DIR"]))

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt

In [ ]:
# Functions
def linear_cka(X, Y):
    """X: [n, d1], Y: [n, d2]. Returns scalar in [0, 1]."""
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    xty = X.T @ Y
    num = (xty ** 2).sum()
    den = np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)
    return float(num / den)


def cka_matrix(A, B):
    M = np.zeros((A.shape[0], B.shape[0]))
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            M[i, j] = linear_cka(A[i], B[j])
    return M


def main():
    data = np.load(str(DATA_DIR / "activations.npz"))
    A, B, R = data["A_layers"], data["B_layers"], data["R_layers"]

    print("Computing CKA: GPT-2 vs Pythia...")
    M_trained = cka_matrix(A, B)
    print("Computing CKA baseline: GPT-2 vs random-weights Pythia...")
    M_random = cka_matrix(A, R)

    diag = np.diag(M_trained[: min(M_trained.shape), : min(M_trained.shape)])
    print(f"\nTrained-vs-trained: mean={M_trained.mean():.3f}, "
          f"diag mean={diag.mean():.3f}, max={M_trained.max():.3f}")
    print(f"Trained-vs-random:  mean={M_random.mean():.3f}, "
          f"max={M_random.max():.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, M, title in [
        (axes[0], M_trained, "GPT-2 vs Pythia-160M (both trained)"),
        (axes[1], M_random, "GPT-2 vs Pythia (random weights)"),
    ]:
        im = ax.imshow(M, vmin=0, vmax=1, cmap="magma", origin="lower")
        ax.set_xlabel("Pythia layer")
        ax.set_ylabel("GPT-2 layer")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "cka_matrix.png"), dpi=150)
    print("\nSaved cka_matrix.png")
    print("Hypothesis supported if left panel shows a hot diagonal "
          "and right panel is uniformly cold.")

In [ ]:
# Run the analysis (requires activations.npz from stage A1)
main()